# ✈️ FlightSense Pro — Diwali 2025 Flight Price Predictor
### Multi-Year ML + Confidence Intervals + Competitor Comparison + Booking Engine
**Routes:** BLR → Ranchi (IXR) | BLR → Lucknow (LKO)  
**Diwali 2025:** October 20, 2025

| Feature | Description |
|---|---|
| 🌐 Live Data | Amadeus API with local cache |
| 📊 Confidence Intervals | 90% bootstrap uncertainty bands |
| 🏆 Competitor Comparison | IndiGo vs Air India vs SpiceJet vs Akasa |
| 🤖 Booking Engine | BUY NOW / WAIT / SET ALERT signals |

In [ ]:
# Run this cell first to install all dependencies
import subprocess, sys

packages = [
    'pandas', 'numpy', 'matplotlib', 'scikit-learn',
    'scipy', 'xgboost', 'lightgbm', 'requests'
]
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '⚠️'
    print(f'{status} {pkg}')

print('\nAll dependencies ready!')

In [ ]:
import os, time, math, json, warnings, hashlib
from datetime import datetime, timedelta, date
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from pathlib import Path
from enum import Enum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.dates import DateFormatter
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.utils import resample

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False
})

def try_import(name):
    try: return __import__(name)
    except ImportError: return None

xgb_mod = try_import('xgboost')
lgb_mod  = try_import('lightgbm')
req_mod  = try_import('requests')

print(f'XGBoost : {"✅" if xgb_mod else "⚠️  using sklearn fallback"}')
print(f'LightGBM: {"✅" if lgb_mod else "⚠️  using sklearn fallback"}')
print(f'Requests: {"✅" if req_mod else "❌  needed for live mode"}')
print('\nImports complete!')

## ⚙️ Configuration — Edit This Cell

In [ ]:
# ════════════════════════════════════════════════
#  ✏️  EDIT THIS CELL TO CUSTOMISE YOUR RUN
# ════════════════════════════════════════════════

# ── Data mode ─────────────────────────────────
# 'simulate' → no API needed, uses realistic synthetic data
# 'live'     → fetches real prices from Amadeus API
DATA_MODE = 'simulate'

# ── Amadeus API credentials (only needed for live mode) ───
AMADEUS_ID     = os.environ.get('AMADEUS_CLIENT_ID',     'YOUR_CLIENT_ID')
AMADEUS_SECRET = os.environ.get('AMADEUS_CLIENT_SECRET', 'YOUR_SECRET')
AMADEUS_ENV    = 'test'   # change to 'production' when approved

# ── Routes ────────────────────────────────────
PRIMARY_ROUTES = [
    ('BLR', 'IXR', 'Bengaluru → Ranchi'),
    ('BLR', 'LKO', 'Bengaluru → Lucknow'),
]
AIRLINES = {'6E': 'IndiGo', 'AI': 'Air India', 'SG': 'SpiceJet', 'QP': 'Akasa'}

# ── Date window ───────────────────────────────
DIWALI_2025  = date(2025, 10, 20)
WINDOW_START = date(2025, 10,  1)
WINDOW_END   = date(2025, 11, 10)
HIST_YEARS   = [2022, 2023, 2024]
DIWALI_HIST  = {2022: date(2022,10,24), 2023: date(2023,11,12), 2024: date(2024,11,1)}

# ── Model settings ────────────────────────────
TRAIN_RATIO  = 0.80
N_BOOTSTRAP  = 80     # increase to 200 for tighter CI (slower)
CI_LEVEL     = 0.90   # 90% confidence interval
CURRENT_LF   = 0.35   # current seat load factor (0-1)
TARGET_LF    = 0.70

# ── Booking thresholds ────────────────────────
BUY_THRESHOLD  = 0.05   # within 5% of predicted min → BUY NOW
WAIT_THRESHOLD = 0.15   # >15% above predicted min  → WAIT

# ── Output folder ─────────────────────────────
OUTDIR = Path('flightsense_outputs')
OUTDIR.mkdir(exist_ok=True)
CACHE  = OUTDIR / 'price_cache'
CACHE.mkdir(exist_ok=True)

print(f'Mode     : {DATA_MODE}')
print(f'Routes   : {[r[2] for r in PRIMARY_ROUTES]}')
print(f'Window   : {WINDOW_START} → {WINDOW_END}')
print(f'Bootstrap: {N_BOOTSTRAP} iters  CI={CI_LEVEL:.0%}')
print(f'Output   : {OUTDIR.resolve()}')

## 📡 Data Layer — Amadeus API + Simulation

In [ ]:
@dataclass
class AmadeusClient:
    client_id: str
    client_secret: str
    env: str = 'test'
    _token: Optional[str] = field(default=None, repr=False)
    _expires: float = field(default=0.0, repr=False)

    def _base(self):
        return ('https://test.api.amadeus.com'
                if self.env != 'production' else 'https://api.amadeus.com')

    def _auth(self):
        if self._token and time.time() < self._expires - 30:
            return
        r = req_mod.post(
            f'{self._base()}/v1/security/oauth2/token',
            data={'grant_type': 'client_credentials',
                  'client_id': self.client_id,
                  'client_secret': self.client_secret},
            timeout=30)
        r.raise_for_status()
        d = r.json()
        self._token   = d['access_token']
        self._expires = time.time() + int(d.get('expires_in', 1800))
        print('  🔐 Token refreshed')

    def fetch_day(self, origin, dest, dep_date, currency='INR', max_offers=50):
        self._auth()
        r = req_mod.get(
            f'{self._base()}/v2/shopping/flight-offers',
            headers={'Authorization': f'Bearer {self._token}'},
            params={'originLocationCode': origin,
                    'destinationLocationCode': dest,
                    'departureDate': dep_date,
                    'adults': 1, 'currencyCode': currency,
                    'max': max_offers},
            timeout=45)
        if r.status_code == 429:
            print('  ⏳ Rate limited — waiting 3s')
            time.sleep(3)
        r.raise_for_status()
        return r.json().get('data', [])


def parse_offers(offers):
    result = {}
    for offer in offers:
        price = float(offer.get('price', {}).get('total', 0) or 0)
        if not price:
            continue
        try:
            carrier = offer['itineraries'][0]['segments'][0]['carrierCode']
        except (KeyError, IndexError):
            carrier = 'XX'
        if carrier not in result or price < result[carrier]:
            result[carrier] = price
    if result:
        result['ALL'] = min(result.values())
    return result


def fetch_live(client, origin, dest, start, end):
    rows = []
    d = start
    while d <= end:
        cache_file = CACHE / f'{origin}_{dest}_{d}.json'
        if cache_file.exists():
            parsed = json.loads(cache_file.read_text())
        else:
            try:
                offers = client.fetch_day(origin, dest, str(d))
                parsed = parse_offers(offers)
                cache_file.write_text(json.dumps(parsed))
                print(f'  📡 {d}  lowest=₹{parsed.get("ALL","N/A")}')
                time.sleep(0.3)
            except Exception as e:
                parsed = {}
                print(f'  ❌ {d} error: {e}')
        row = {'date': d, 'lowest_price': parsed.get('ALL')}
        for code in AIRLINES:
            row[f'price_{code}'] = parsed.get(code)
        rows.append(row)
        d += timedelta(days=1)
    return pd.DataFrame(rows)


def simulate_year(origin, dest, year, diwali_date):
    seed = int(hashlib.md5(f'{origin}{dest}{year}'.encode()).hexdigest()[:6], 16)
    rng  = np.random.default_rng(seed)
    base = {'IXR': 4200, 'LKO': 4600}.get(dest, 4800) * (1.08 ** (year - 2022))
    dm   = {2022: 1.10, 2023: 0.92, 2024: 1.05}.get(year, 1.0)
    rows = []
    d = diwali_date - timedelta(days=45)
    while d <= diwali_date + timedelta(days=30):
        dtd = max((diwali_date - d).days, 0)
        dfd = (d - diwali_date).days
        surge    = 1 + 1.8 * math.exp(-0.08 * dtd) * dm
        ret_rush = 0.45 * math.exp(-0.3 * abs(dfd - 5)) if 3 <= dfd <= 7 else 0
        earlybird= 0.88 if dtd > 30 else 1.0
        weekend  = 0.13 if d.weekday() >= 4 else 0
        p = max(base * 0.65,
                base * (surge + ret_rush) * (1 + weekend) * earlybird
                + rng.normal(0, base * 0.035))
        row = {'date': d, 'year': year, 'lowest_price': round(p, 2)}
        row['price_6E'] = round(p * rng.uniform(0.95, 1.02), 2)
        row['price_AI'] = round(p * rng.uniform(1.05, 1.18), 2)
        row['price_SG'] = round(p * rng.uniform(0.97, 1.08), 2)
        row['price_QP'] = round(p * rng.uniform(0.93, 1.05), 2)
        rows.append(row)
        d += timedelta(days=1)
    df = pd.DataFrame(rows)
    df['date'] = pd.to_datetime(df['date'])
    return df


def get_data(origin, dest):
    # Historical years
    hist_frames = []
    for yr in HIST_YEARS:
        df = simulate_year(origin, dest, yr, DIWALI_HIST[yr])
        hist_frames.append(df)
        print(f'  {yr}: {len(df)} days  ₹{df.lowest_price.min():,.0f}–₹{df.lowest_price.max():,.0f}')
    hist = pd.concat(hist_frames, ignore_index=True)

    # 2025 window
    if DATA_MODE == 'live' and req_mod and AMADEUS_ID != 'YOUR_CLIENT_ID':
        print('  Fetching live 2025 data...')
        client = AmadeusClient(AMADEUS_ID, AMADEUS_SECRET, AMADEUS_ENV)
        curr = fetch_live(client, origin, dest, WINDOW_START, WINDOW_END)
    else:
        if DATA_MODE == 'live':
            print('  ⚠️  Live mode but no credentials — using simulation')
        curr = simulate_year(origin, dest, 2025, DIWALI_2025)
        curr = curr[
            (curr.date >= pd.Timestamp(WINDOW_START)) &
            (curr.date <= pd.Timestamp(WINDOW_END))
        ].reset_index(drop=True)
        print(f'  2025: {len(curr)} days  ₹{curr.lowest_price.min():,.0f}–₹{curr.lowest_price.max():,.0f}')

    return hist, curr


print('✅ Data layer ready')

## 🔧 Feature Engineering

In [ ]:
FEATURE_COLS = [
    'days_to_diwali', 'days_after_diwali',
    'diwali_surge_exp', 'diwali_surge_quad',
    'return_rush', 'is_holiday_window',
    'day_of_week', 'is_weekend', 'days_to_month_end',
    'load_factor', 'lf_shortfall', 'lf_urgency', 'days_ahead',
    'price_lag_1', 'price_lag_3', 'price_lag_7', 'price_lag_14',
    'price_roll3', 'price_roll7', 'price_roll14', 'price_std7',
]

def engineer(df_raw, diwali_date=DIWALI_2025, lf=CURRENT_LF):
    df  = df_raw.copy()
    df['date'] = pd.to_datetime(df['date'])
    df  = df.sort_values('date').reset_index(drop=True)
    dts = pd.Timestamp(diwali_date)
    today = pd.Timestamp('2025-09-01')

    df['days_to_diwali']    = (dts - df['date']).dt.days.clip(lower=0)
    df['days_after_diwali'] = (df['date'] - dts).dt.days.clip(lower=0)
    df['diwali_surge_exp']  = np.exp(-0.08 * df['days_to_diwali'])
    df['diwali_surge_quad'] = 1 / (1 + 0.01 * df['days_to_diwali'] ** 2)
    df['return_rush']       = (
        np.exp(-0.3 * (df['days_after_diwali'] - 5).abs()) *
        df['days_after_diwali'].between(3, 7).astype(float))
    df['is_holiday_window'] = df['date'].between(
        dts - pd.Timedelta(days=7), dts + pd.Timedelta(days=5)).astype(int)
    df['day_of_week']       = df['date'].dt.dayofweek
    df['is_weekend']        = (df['day_of_week'] >= 4).astype(int)
    df['days_to_month_end'] = df['date'].dt.days_in_month - df['date'].dt.day
    df['days_ahead']        = (df['date'] - today).dt.days.clip(lower=0)
    lf_v = float(lf)
    df['load_factor']  = lf_v
    df['lf_shortfall'] = max(0.0, TARGET_LF - lf_v)
    df['lf_urgency']   = df['lf_shortfall'] * df['diwali_surge_exp']
    p = df['lowest_price'].ffill().bfill()
    for lag in [1, 3, 7, 14]:
        df[f'price_lag_{lag}'] = p.shift(lag)
    df['price_roll3']  = p.rolling(3,  min_periods=1).mean()
    df['price_roll7']  = p.rolling(7,  min_periods=1).mean()
    df['price_roll14'] = p.rolling(14, min_periods=1).mean()
    df['price_std7']   = p.rolling(7,  min_periods=2).std().fillna(0)
    return df.bfill().ffill()

print(f'✅ {len(FEATURE_COLS)} features defined')

## 🤖 Models + Bootstrap Confidence Intervals

In [ ]:
def build_gbm(X, y):
    if lgb_mod:
        m = lgb_mod.LGBMRegressor(
            n_estimators=400, learning_rate=0.04,
            num_leaves=31, subsample=0.8,
            colsample_bytree=0.8, random_state=42, verbose=-1)
    elif xgb_mod:
        m = xgb_mod.XGBRegressor(
            n_estimators=400, learning_rate=0.04,
            max_depth=5, subsample=0.8,
            colsample_bytree=0.8, random_state=42, verbosity=0)
    else:
        m = GradientBoostingRegressor(
            n_estimators=400, learning_rate=0.04,
            max_depth=5, subsample=0.8, random_state=42)
    m.fit(X, y)
    return m


def holt_winters(y, s=7):
    """Triple exponential smoothing — SARIMA proxy."""
    L = y[:s].mean()
    T = (y[s:2*s].mean() - y[:s].mean()) / s
    S = list(y[:s] - L)
    a, b, g = 0.25, 0.08, 0.15
    Ls, Ts, Ss = L, T, list(S)
    for i in range(len(y)):
        obs = y[i]; Lp, Tp, Sp = Ls, Ts, Ss[i % s]
        Ls = a*(obs-Sp) + (1-a)*(Lp+Tp)
        Ts = b*(Ls-Lp) + (1-b)*Tp
        Ss[i%s] = g*(obs-Ls) + (1-g)*Sp
    n = len(y)
    return lambda h: [Ls + k*Ts + Ss[(n+k-1)%s] for k in range(1, h+1)]


def build_neural(X, y):
    scX = MinMaxScaler(); scY = MinMaxScaler()
    Xs  = scX.fit_transform(X)
    ys  = scY.fit_transform(y.reshape(-1,1)).flatten()
    m   = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), max_iter=800,
        learning_rate_init=0.001, early_stopping=True,
        validation_fraction=0.15, random_state=42, alpha=1e-4)
    m.fit(Xs, ys)
    return m, scX, scY


def bootstrap_ci(X_tr, y_tr, X_te, n_boot=N_BOOTSTRAP, ci=CI_LEVEL):
    """Bootstrap resampling for prediction uncertainty bands."""
    print(f'  Running {n_boot} bootstrap iterations...')
    boot  = []
    alpha = (1 - ci) / 2
    for i in range(n_boot):
        Xb, yb = resample(X_tr, y_tr, random_state=i)
        m = build_gbm(Xb, yb)
        boot.append(np.clip(m.predict(X_te), 0, None))
        if (i+1) % 20 == 0:
            print(f'    {i+1}/{n_boot} done')
    arr = np.array(boot)
    return (arr.mean(axis=0),
            np.percentile(arr, alpha * 100,       axis=0),
            np.percentile(arr, (1-alpha) * 100,   axis=0))


print('✅ Models ready')

## 🎯 Booking Recommendation Engine

In [ ]:
class Signal(str, Enum):
    BUY_NOW   = 'BUY_NOW'
    WAIT      = 'WAIT'
    SET_ALERT = 'SET_ALERT'
    UNCERTAIN = 'UNCERTAIN'

SIGNAL_EMOJI = {
    Signal.BUY_NOW:   '🟢 BUY NOW',
    Signal.WAIT:      '🔴 WAIT',
    Signal.SET_ALERT: '🟡 SET ALERT',
    Signal.UNCERTAIN: '⚪ UNCERTAIN',
}

PALETTE = {
    'Actual': '#1a1a2e',  'GBM':    '#e67e22',
    'SARIMA': '#3498db',  'Neural': '#9b59b6',
    'Ensemble': '#e74c3c',
    'IndiGo': '#0056b3',  'Air India': '#ff6b35',
    'SpiceJet': '#ff2800','Akasa': '#00b4d8',
    Signal.BUY_NOW:   '#27ae60',
    Signal.WAIT:      '#e74c3c',
    Signal.SET_ALERT: '#f39c12',
    Signal.UNCERTAIN: '#95a5a6',
}


def make_recommendation(route, dep_date, current_price,
                         ens_preds, ci_lo, ci_hi, row, days_to_diwali):
    pred_min  = float(np.percentile(ens_preds, 10))
    pred_max  = float(np.percentile(ens_preds, 90))
    ci_width  = (ci_hi - ci_lo) / max(current_price, 1)
    ratio     = current_price / max(pred_min, 1)
    confidence= float(np.clip(1 - ci_width, 0, 1))

    # Airline prices
    ap = {}
    for code, name in AIRLINES.items():
        col = f'price_{code}'
        if col in row.index and pd.notna(row[col]):
            ap[name] = float(row[col])
    cheapest = min(ap, key=ap.get) if ap else 'N/A'

    if days_to_diwali < 7:
        signal = Signal.UNCERTAIN
        reason = (f'Only {days_to_diwali} days to Diwali. '
                  'Prices rarely drop this close — book if you need to.')
    elif ratio <= (1 + BUY_THRESHOLD) and confidence >= 0.60:
        signal = Signal.BUY_NOW
        reason = (f'₹{current_price:,.0f} is within {BUY_THRESHOLD:.0%} of '
                  f'predicted floor ₹{pred_min:,.0f}. '
                  f'Confidence: {confidence:.0%}. Book now!')
    elif ratio > (1 + WAIT_THRESHOLD) and days_to_diwali >= 14:
        signal = Signal.WAIT
        saving = current_price - pred_min
        reason = (f'₹{current_price:,.0f} is {ratio-1:.0%} above predicted '
                  f'minimum ₹{pred_min:,.0f}. '
                  f'Expected saving if you wait: ₹{saving:,.0f}.')
    elif ci_width > 0.20:
        signal = Signal.SET_ALERT
        reason = (f'Price is near the floor but uncertainty is high '
                  f'(CI width {ci_width:.0%}). Set alert at ₹{pred_min:,.0f}.')
    else:
        signal = Signal.BUY_NOW
        reason = (f'₹{current_price:,.0f} is in an acceptable range. '
                  f'Confidence: {confidence:.0%}.')

    return {
        'route':           route,
        'date':            str(dep_date),
        'signal':          signal.value,
        'confidence':      round(confidence, 3),
        'current_price':   round(current_price, 2),
        'predicted_min':   round(pred_min, 2),
        'predicted_max':   round(pred_max, 2),
        'ci_lower':        round(float(ci_lo), 2),
        'ci_upper':        round(float(ci_hi), 2),
        'reason':          reason,
        'days_to_diwali':  int(days_to_diwali),
        'cheapest_airline':cheapest,
        'airline_prices':  ap,
        'generated_at':    datetime.utcnow().isoformat() + 'Z',
    }

print('✅ Booking engine ready')

## 🚀 Run Full Pipeline

In [ ]:
def run_pipeline(origin, dest, label):
    print(f'\n{"="*60}')
    print(f'  ✈️  {label}  ({origin} → {dest})')
    print(f'{"="*60}')

    # 1. Data
    print('\n[DATA] Loading...')
    hist, curr = get_data(origin, dest)
    hist['date'] = pd.to_datetime(hist['date'])
    curr['date'] = pd.to_datetime(curr['date'])

    # 2. Feature engineering
    hist_fe = pd.concat([
        engineer(grp, diwali_date=DIWALI_HIST[yr])
        for yr, grp in hist.groupby('year')
    ], ignore_index=True)
    curr_fe = engineer(curr, diwali_date=DIWALI_2025)

    # 3. Train / test split
    n     = len(curr_fe)
    split = int(n * TRAIN_RATIO)
    tr25, te25 = curr_fe.iloc[:split], curr_fe.iloc[split:]

    X_tr = np.vstack([
        hist_fe[FEATURE_COLS].values,
        tr25[FEATURE_COLS].values
    ])
    y_tr = np.concatenate([
        hist_fe['lowest_price'].values,
        tr25['lowest_price'].values
    ])
    X_te = te25[FEATURE_COLS].values
    y_te = te25['lowest_price'].values
    print(f'  Training: {len(X_tr):,} samples  |  Test: {len(y_te)} samples')

    # 4. Models
    all_preds = {}; errors = {}

    print('\n[GBM] Fitting...')
    gbm = build_gbm(X_tr, y_tr)
    p   = np.clip(gbm.predict(X_te), 0, None)
    all_preds['GBM'] = p
    errors['GBM']    = math.sqrt(mean_squared_error(y_te, p))
    feat_imp = pd.Series(gbm.feature_importances_,
                          index=FEATURE_COLS).sort_values(ascending=False)
    print(f'  RMSE = ₹{errors["GBM"]:,.0f}')

    print('\n[SARIMA] Fitting...')
    hw_fc = holt_winters(y_tr[-90:])
    p     = np.clip(hw_fc(len(y_te)), 0, None)
    all_preds['SARIMA'] = p
    errors['SARIMA']    = math.sqrt(mean_squared_error(y_te, p))
    print(f'  RMSE = ₹{errors["SARIMA"]:,.0f}')

    print('\n[Neural Net] Fitting...')
    nn, scX, scY = build_neural(X_tr, y_tr)
    p = np.clip(scY.inverse_transform(
        nn.predict(scX.transform(X_te)).reshape(-1,1)).flatten(), 0, None)
    all_preds['Neural'] = p
    errors['Neural']    = math.sqrt(mean_squared_error(y_te, p))
    print(f'  RMSE = ₹{errors["Neural"]:,.0f}')

    # Ensemble (inverse-RMSE weights)
    inv = {k: 1/max(v, 1e-6) for k,v in errors.items()}
    s   = sum(inv.values())
    wts = {k: v/s for k,v in inv.items()}
    ens = sum(all_preds[k] * wts[k] for k in all_preds)
    errors['Ensemble'] = math.sqrt(mean_squared_error(y_te, ens))
    print(f'\n[Ensemble] RMSE = ₹{errors["Ensemble"]:,.0f}')
    print(f'  Weights: {({k: f"{v:.0%}" for k,v in wts.items()})}')

    # 5. Bootstrap confidence intervals
    print()
    ci_mean, ci_lo, ci_hi = bootstrap_ci(X_tr, y_tr, X_te)

    # 6. Booking recommendations
    recs = []
    for idx, (i, row) in enumerate(te25.iterrows()):
        rec = make_recommendation(
            route         = label,
            dep_date      = row['date'].date(),
            current_price = row['lowest_price'],
            ens_preds     = ens,
            ci_lo         = ci_lo[idx],
            ci_hi         = ci_hi[idx],
            row           = row,
            days_to_diwali= int(row['days_to_diwali']),
        )
        recs.append(rec)

    # 7. Competitor summary
    comp = {}
    for code, name in AIRLINES.items():
        col = f'price_{code}'
        if col in curr_fe.columns:
            vals = curr_fe[col].dropna()
            if len(vals):
                comp[name] = {
                    'min':  round(float(vals.min()), 2),
                    'mean': round(float(vals.mean()), 2),
                    'max':  round(float(vals.max()), 2),
                }

    return dict(
        origin=origin, dest=dest, label=label,
        curr_fe=curr_fe, tr25=tr25, te25=te25,
        y_tr=y_tr, y_te=y_te,
        all_preds=all_preds, ens=ens,
        ci_lo=ci_lo, ci_hi=ci_hi,
        errors=errors, weights=wts,
        feat_imp=feat_imp, recs=recs, comp=comp,
    )

# ── Run both routes ────────────────────────────────────────────────────────────
RESULTS = {}
for origin, dest, label in PRIMARY_ROUTES:
    RESULTS[dest] = run_pipeline(origin, dest, label)

print('\n✅ Pipeline complete!')

## 📊 Charts

In [ ]:
def plot_route(R):
    cf, tr, te = R['curr_fe'], R['tr25'], R['te25']
    label = R['label']
    origin, dest = R['origin'], R['dest']
    dates_all = cf['date'].values
    dates_te  = te['date'].values
    prices    = cf['lowest_price'].values
    y_te      = R['y_te']

    fig = plt.figure(figsize=(18, 14))
    fig.patch.set_facecolor('#fafafa')
    gs  = gridspec.GridSpec(3, 3, figure=fig,
                            hspace=0.48, wspace=0.32,
                            top=0.91, bottom=0.05)

    # Panel 1: Main forecast + CI band
    ax1 = fig.add_subplot(gs[0, :])
    ax1.set_facecolor('#ffffff')
    ax1.axvspan(pd.Timestamp(dates_all[0]),
                pd.Timestamp(tr['date'].values[-1]),
                alpha=0.04, color='steelblue')
    ax1.plot(dates_all, prices, color=PALETTE['Actual'],
             lw=2.2, marker='o', ms=3.5, label='Observed', zorder=5)
    ax1.fill_between(dates_te, R['ci_lo'], R['ci_hi'],
                     alpha=0.22, color=PALETTE['Ensemble'],
                     label=f'{CI_LEVEL:.0%} Confidence Band')
    ls_map = {'GBM': '-.', 'SARIMA': '--', 'Neural': ':'}
    for mn, pred in R['all_preds'].items():
        ax1.plot(dates_te, pred, color=PALETTE[mn], lw=1.5, alpha=0.7,
                 linestyle=ls_map.get(mn, '-'),
                 label=f'{mn}  RMSE ₹{R["errors"][mn]:,.0f}')
    ax1.plot(dates_te, R['ens'], color=PALETTE['Ensemble'], lw=3.0,
             label=f'Ensemble  RMSE ₹{R["errors"]["Ensemble"]:,.0f}', zorder=4)

    # Booking signal dots
    for rec in R['recs']:
        clr = PALETTE.get(Signal(rec['signal']), '#aaa')
        ax1.scatter(pd.Timestamp(rec['date']), rec['current_price'],
                    s=65, color=clr, zorder=6, edgecolors='white', lw=0.8)

    dts = pd.Timestamp(DIWALI_2025)
    ax1.axvline(dts, color='#c0392b', lw=1.8, ls='--', alpha=0.85)
    ax1.annotate(' 🪔 Diwali\n Oct 20', xy=(dts, prices.max()),
                 fontsize=9, color='#c0392b', va='top')
    ax1.axvspan(dts - pd.Timedelta(days=7),
                dts + pd.Timedelta(days=5), alpha=0.06, color='#e74c3c')
    ax1.xaxis.set_major_formatter(DateFormatter('%b %d'))
    sig_patches = [mpatches.Patch(color=PALETTE[s], label=SIGNAL_EMOJI[s])
                   for s in Signal]
    h, l = ax1.get_legend_handles_labels()
    ax1.legend(h + sig_patches, l + [SIGNAL_EMOJI[s] for s in Signal],
               fontsize=7.5, loc='upper left', framealpha=0.92, ncol=3)
    ax1.set_title(f'{label} — Ensemble Forecast + {CI_LEVEL:.0%} Confidence Band',
                  fontsize=12, fontweight='bold', pad=8)
    ax1.set_ylabel('Price (₹)', fontsize=10)
    ax1.grid(True, alpha=0.2, ls='--')

    # Panel 2: Competitor airlines
    ax2 = fig.add_subplot(gs[1, :2])
    ax2.set_facecolor('#ffffff')
    for code, name in AIRLINES.items():
        col = f'price_{code}'
        if col in cf.columns:
            ax2.plot(dates_all, cf[col].ffill().bfill().values,
                     color=PALETTE.get(name, '#95a5a6'), lw=1.8,
                     label=name, alpha=0.85)
    ax2.plot(dates_all, prices, color=PALETTE['Actual'],
             lw=2.2, ls='--', label='Best Available', alpha=0.55)
    ax2.axvline(dts, color='#c0392b', lw=1.5, ls='--', alpha=0.7)
    ax2.xaxis.set_major_formatter(DateFormatter('%b %d'))
    ax2.set_title('Competitor Airline Price Comparison',
                  fontsize=10, fontweight='bold')
    ax2.set_ylabel('Price (₹)', fontsize=9)
    ax2.legend(fontsize=8.5); ax2.grid(True, alpha=0.2, ls='--')

    # Panel 3: Signal pie
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.set_facecolor('#ffffff')
    sv = {s: 0 for s in Signal}
    for rec in R['recs']:
        sv[Signal(rec['signal'])] += 1
    colors = [PALETTE.get(s, '#ccc') for s in sv]
    labels = [f'{SIGNAL_EMOJI[s]}\n({v} days)' for s, v in sv.items()]
    ax3.pie(list(sv.values()), colors=colors, startangle=90,
            labels=labels,
            wedgeprops={'edgecolor': 'white', 'lw': 2},
            textprops={'fontsize': 7.5})
    ax3.set_title('Booking Signal Breakdown', fontsize=10, fontweight='bold')

    # Panel 4: Uncertainty timeline
    ax4 = fig.add_subplot(gs[2, :2])
    ax4.set_facecolor('#ffffff')
    ci_pct = (R['ci_hi'] - R['ci_lo']) / R['ens'] * 100
    ax4.fill_between(dates_te, 0, ci_pct, alpha=0.30, color='#e74c3c')
    ax4.plot(dates_te, ci_pct, color='#c0392b', lw=2)
    ax4.axhline(20, color='#f39c12', lw=1.2, ls='--',
                label='High uncertainty (20%)')
    ax4.axhline(10, color='#27ae60', lw=1.2, ls='--',
                label='Low uncertainty (10%)')
    ax4.xaxis.set_major_formatter(DateFormatter('%b %d'))
    ax4.set_title(f'Prediction Uncertainty Over Time ({CI_LEVEL:.0%} CI Width)',
                  fontsize=10, fontweight='bold')
    ax4.set_ylabel('CI Width %', fontsize=9)
    ax4.legend(fontsize=8); ax4.grid(True, alpha=0.2, ls='--')

    # Panel 5: Feature importance
    ax5 = fig.add_subplot(gs[2, 2])
    ax5.set_facecolor('#ffffff')
    top = R['feat_imp'].head(10)
    ax5.barh(top.index[::-1], top.values[::-1],
             color='#e67e22', edgecolor='white', lw=1)
    ax5.set_title('Top 10 Feature Importances', fontsize=10, fontweight='bold')
    ax5.set_xlabel('Importance', fontsize=9)
    ax5.grid(True, axis='x', alpha=0.2, ls='--')

    fig.suptitle(
        f'✈️  FlightSense Pro  ·  {label}  ·  '
        f'Load Factor={CURRENT_LF:.0%}  ·  CI={CI_LEVEL:.0%}  ·  '
        f'Training: {len(HIST_YEARS)} historical years + 2025',
        fontsize=12.5, fontweight='bold', y=0.97)

    out = OUTDIR / f'{origin}_{dest}_flightsense_pro.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'  Chart saved → {out}')


for dest, R in RESULTS.items():
    plot_route(R)

## 📋 Booking Recommendations

In [ ]:
for dest, R in RESULTS.items():
    recs  = R['recs']
    label = R['label']
    print(f'\n{"="*72}')
    print(f'  📋  {label}')
    print(f'{"="*72}')
    print(f'  {"Date":<12} {"Price":>8} {"Pred Min":>9} '
          f'{"CI Lower":>9} {"CI Upper":>9}  {"Signal":<16} Cheapest Airline')
    print(f'  {"-"*70}')
    for rec in recs:
        sig = SIGNAL_EMOJI[Signal(rec['signal'])]
        print(f'  {rec["date"]:<12} '
              f'₹{rec["current_price"]:>7,.0f} '
              f'₹{rec["predicted_min"]:>8,.0f} '
              f'₹{rec["ci_lower"]:>8,.0f} '
              f'₹{rec["ci_upper"]:>8,.0f}  '
              f'{sig:<17} {rec["cheapest_airline"]}')

    buy  = [r for r in recs if r['signal'] == 'BUY_NOW']
    wait = [r for r in recs if r['signal'] == 'WAIT']
    if buy:
        best = min(buy, key=lambda r: r['current_price'])
        print(f'\n  ✅ BEST DATE TO BOOK: {best["date"]}  '
              f'₹{best["current_price"]:,.0f}  via {best["cheapest_airline"]}')
        print(f'     {best["reason"]}')

    cs = R['comp']
    if cs:
        print(f'\n  🏆 AIRLINE COMPARISON:')
        for airline, s in sorted(cs.items(), key=lambda x: x[1]['mean']):
            print(f'     {airline:<12}  '
                  f'min ₹{s["min"]:>7,.0f}  '
                  f'avg ₹{s["mean"]:>7,.0f}  '
                  f'max ₹{s["max"]:>7,.0f}')
        cheapest = min(cs, key=lambda a: cs[a]['mean'])
        priciest = max(cs, key=lambda a: cs[a]['mean'])
        saving   = cs[priciest]['mean'] - cs[cheapest]['mean']
        print(f'\n  💰 Switch from {priciest} to {cheapest}: save ₹{saving:,.0f}/ticket')

## 💾 Export — JSON + CSV

In [ ]:
# Master JSON (API-ready)
master = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'diwali_2025':  str(DIWALI_2025),
    'data_mode':    DATA_MODE,
    'ci_level':     CI_LEVEL,
    'routes': {
        R['label']: {
            'model_errors':       R['errors'],
            'ensemble_weights':   {k: round(v,4) for k,v in R['weights'].items()},
            'competitor_summary': R['comp'],
            'recommendations':    R['recs'],
        }
        for dest, R in RESULTS.items()
    }
}
json_path = OUTDIR / 'flightsense_output.json'
json_path.write_text(json.dumps(master, indent=2))
print(f'✅ JSON saved → {json_path}')

# CSV
all_recs = []
for dest, R in RESULTS.items():
    for rec in R['recs']:
        flat = {k: v for k, v in rec.items()
                if k not in ('airline_prices', 'generated_at')}
        all_recs.append(flat)

csv_path = OUTDIR / 'recommendations.csv'
pd.DataFrame(all_recs).to_csv(csv_path, index=False)
print(f'✅ CSV  saved → {csv_path}')
print(f'\n{len(all_recs)} total recommendations across {len(RESULTS)} routes')
pd.DataFrame(all_recs)[['route','date','signal','confidence',
                          'current_price','predicted_min','cheapest_airline']]

## 🚀 Integration Guide

### Switch to Live Data
In the **Configuration cell**, change:
```python
DATA_MODE = 'live'
AMADEUS_ID     = 'your_client_id'
AMADEUS_SECRET = 'your_secret'
```
Prices are cached in `flightsense_outputs/price_cache/` — same date never fetched twice.

### Daily Auto-Run (cron)
```bash
# Mac/Linux — runs every morning at 7 AM
0 7 * * * cd /path/to/project && jupyter nbconvert --to notebook --execute flight_tracker_pro.ipynb
```

### Webhook on BUY signal
```python
import requests
for rec in recommendations:
    if rec['signal'] == 'BUY_NOW' and rec['confidence'] > 0.70:
        requests.post('https://your-app.com/webhook/price-alert',
                      json=rec)
```

### API Response Shape (from flightsense_output.json)
```json
{
  "signal": "BUY_NOW",
  "confidence": 0.82,
  "current_price": 6840,
  "predicted_min": 6612,
  "ci_lower": 6280,
  "ci_upper": 7120,
  "cheapest_airline": "Akasa",
  "reason": "Price is within 5% of predicted floor. Book now!"
}
```